<a href="https://colab.research.google.com/github/pachir1su/file_swordmaster/blob/main/ALL_swordmaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio pypdf pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 5.1 MB/s eta 0:00:00


In [3]:
import gradio as gr
from pypdf import PdfReader, PdfWriter
from pydub import AudioSegment
from pydub.silence import split_on_silence
import os

# ==========================================
# 1. PDF 다중 범위 추출 기능 (PDF Swordmaster)
# ==========================================
def parse_page_string(page_string, total_pages):
    """ '1, 3, 5-10' 형태의 문자열을 파싱하여 실제 페이지 번호 리스트로 반환 """
    pages = set()
    if not page_string:
        return []

    for part in page_string.split(','):
        part = part.strip()
        if '-' in part:
            start, end = map(int, part.split('-'))
            pages.update(range(start, end + 1))
        else:
            pages.add(int(part))

    # 유효한 페이지 범위만 정렬하여 반환
    return sorted([p for p in pages if 1 <= p <= total_pages])

def process_pdf(pdf_file, page_string):
    if pdf_file is None:
        return None, "PDF 파일을 업로드해주세요."

    try:
        reader = PdfReader(pdf_file.name)
        total_pages = len(reader.pages)
        target_pages = parse_page_string(page_string, total_pages)

        if not target_pages:
            return None, f"유효하지 않은 페이지 입력입니다. (총 {total_pages}페이지)"

        writer = PdfWriter()
        for p in target_pages:
            writer.add_page(reader.pages[p - 1]) # 0-index 보정

        output_path = "/content/extracted_output.pdf"
        with open(output_path, "wb") as f:
            writer.write(f)

        return output_path, f"성공적으로 추출되었습니다! (추출된 페이지: {target_pages})"
    except Exception as e:
        return None, f"오류 발생: {str(e)}"

# ==========================================
# 2. 오디오 정밀/무음 타겟팅 기능 (Audio Swordmaster)
# ==========================================
def process_audio(audio_file, start_sec, end_sec, remove_silence):
    if audio_file is None:
        return None, "오디오 파일을 업로드해주세요."

    try:
        # 오디오 불러오기 (Gradio는 임시 파일 경로를 넘겨줌)
        audio = AudioSegment.from_file(audio_file)

        # 1. 정밀 타겟팅 (초 단위 자르기)
        if start_sec > 0 or end_sec > 0:
            start_ms = int(start_sec * 1000)
            end_ms = int(end_sec * 1000) if end_sec > 0 else len(audio)
            audio = audio[start_ms:end_ms]

        # 2. 무음 베어내기 (Auto-trimming)
        if remove_silence:
            # silence_thresh는 환경에 따라 조절 필요 (기본 -40dBFS)
            chunks = split_on_silence(audio, min_silence_len=500, silence_thresh=-40)
            if chunks:
                audio = sum(chunks) # 무음이 제거된 조각들을 다시 합침

        output_path = "/content/processed_audio.mp3"
        audio.export(output_path, format="mp3")

        return output_path, "성공적으로 오디오가 처리되었습니다!"
    except Exception as e:
        return None, f"오류 발생: {str(e)}"

# ==========================================
# 3. Gradio 웹 UI 구성 (Dashboard)
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# File Swordmaster Web UI")
    gr.Markdown("오디오와 PDF 파일을 원하는 대로 정밀하게 분할해 보세요!")

    with gr.Tab("PDF 분할기"):
        with gr.Row():
            with gr.Column():
                pdf_input = gr.File(label="PDF 파일 업로드", file_types=[".pdf"])
                pdf_pages = gr.Textbox(label="추출할 페이지 (예: 1, 3, 5-10)", placeholder="1, 3, 5-10")
                pdf_btn = gr.Button("PDF 자르기", variant="primary")
            with gr.Column():
                pdf_output_file = gr.File(label="결과물 다운로드")
                pdf_output_msg = gr.Textbox(label="상태 메시지", interactive=False)

        pdf_btn.click(process_pdf, inputs=[pdf_input, pdf_pages], outputs=[pdf_output_file, pdf_output_msg])

    with gr.Tab("오디오 분할기"):
        with gr.Row():
            with gr.Column():
                audio_input = gr.Audio(label="오디오 파일 업로드", type="filepath")
                with gr.Row():
                    audio_start = gr.Number(label="시작 시간 (초)", value=0)
                    audio_end = gr.Number(label="종료 시간 (초, 0이면 끝까지)", value=0)
                audio_silence = gr.Checkbox(label="무음 구간 자동 제거 (Auto-trimming)", value=False)
                audio_btn = gr.Button("오디오 자르기", variant="primary")
            with gr.Column():
                audio_output_file = gr.File(label="결과물 다운로드")
                audio_output_msg = gr.Textbox(label="상태 메시지", interactive=False)

        audio_btn.click(process_audio, inputs=[audio_input, audio_start, audio_end, audio_silence], outputs=[audio_output_file, audio_output_msg])

# Colab 환경에서 외부 접속 링크 생성
demo.launch(share=True)

/tmp/ipykernel_3024/2326832310.py:85: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3fa90fd263dd81e569.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
